# Fitbit Fitness Tracker Data - Simple EDA

## Step 0: Setup reference

Environment and input checks are recorded in
[Project_Setup.ipynb](Project_Setup.ipynb).

This notebook will include its own minimal imports and path configuration
so it can run independently of the setup notebook's Python session.

## Step 1: Analysis brief

The purpose, questions, metrics, and analysis rules are recorded in
[analysis_brief.md](analysis_brief.md).

## Notebook setup

Paths, reading rules and small helpers used by every step.

In [1]:
import csv
import hashlib
import re
import sys
from collections import Counter
from pathlib import Path

# Find the project root.
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        folder
        for folder in (cwd, *cwd.parents)
        if (folder / ".git").exists()
        and (folder / "requirements.txt").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

# Stop if the notebook uses the wrong environment.
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Select the project's .venv kernel.")

DATA_ROOT = PROJECT_ROOT / "Dataset"
MANIFEST_PATH = PROJECT_ROOT / "EDA" / "input_manifest.csv"
COUNTS_PATH = PROJECT_ROOT / "EDA" / "distinct_value_counts.csv"

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset folder is missing: {DATA_ROOT}")

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Input manifest is missing: {MANIFEST_PATH}")

# Import pandas after checking the environment.
import pandas as pd 

# Keep original text and avoid automatic conversions (Step 0.4).
READ_POLICY = {
    "sep": ",",
    "encoding": "utf-8-sig",
    "encoding_errors": "strict",
    "dtype": "string",
    "na_filter": False,
    "parse_dates": False,
    "skip_blank_lines": False,
    "on_bad_lines": "error",
}
CHUNK_SIZE = 100_000 

PERSON_ID_COLUMNS = {"Id"}
LOG_ID_COLUMNS = {"logId", "LogId"}
ID_COLUMNS = PERSON_ID_COLUMNS | LOG_ID_COLUMNS

def read_chunks(file_info, columns=None):
    """Read one file in chunks, keeping every value as text."""
    return pd.read_csv(
        DATA_ROOT / file_info.source_file,
        usecols=columns,
        chunksize=CHUNK_SIZE,
        **READ_POLICY,
    )

def percent(part, whole):
    """Percentage with two decimals; NA when the denominator is 0."""
    return round(100 * (part / whole), 2) if whole else pd.NA

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_ROOT)
print("Manifest    :", MANIFEST_PATH)
print("Project environment: OK")

Project root: D:\Projects\AI and ML\FitBit Fitness Tracker Data
Data folder : D:\Projects\AI and ML\FitBit Fitness Tracker Data\Dataset
Manifest    : D:\Projects\AI and ML\FitBit Fitness Tracker Data\EDA\input_manifest.csv
Project environment: OK


## Step 2: Load and inventory the files

Check each file's structure before exploring its values.
Keep the two exports separate and leave raw files unchanged.

### 2.1 File inventory

Start with the saved input manifest.

In [2]:
# Read the saved file list, not the raw datasets.
manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype="string",
    keep_default_na=False,
    encoding="utf-8-sig",
)
manifest["size_bytes"] = pd.to_numeric(manifest["size_bytes"], errors="raise")

summary = manifest.groupby("source_folder").agg(
    files = ("source_file", "size"), 
    size_bytes = ("size_bytes", "sum"), 
)
summary["size_mb"] = (summary["size_bytes"] / 1_000_000).round(1)

print("Files listed", len(manifest))
display(summary)

Files listed 29


,files,size_bytes,size_mb
source_folder,,,
P1,11,278798829,278.8
P2,18,337997716,338.0


#### Check the input files

Check each file's existence, size, and SHA-256 before loading its data.
Stop if anything differs from the saved manifest.

In [3]:
# Flag CSVs on disk that are not in the manifest.
on_disk = {
    path.relative_to(DATA_ROOT).as_posix()
    for path in DATA_ROOT.rglob("*.csv") 
    if path.is_file() 
}
extra = sorted(on_disk - set(manifest["source_file"]))

if extra:
    print("Extra CSVs - not included in this analysis:")
    for name in extra:
        print(name)
else:
    print("No extra CSV files found.")

for entry in manifest.itertuples(index=False): 
    path = DATA_ROOT / entry.source_file 

    if not path.is_file():
        raise FileNotFoundError(f"Missing file: {entry.source_file}")

    if path.stat().st_size != entry.size_bytes:
        raise ValueError(f"File size changed: {entry.source_file}")

    # Hash the file without loading the table into memory.
    with path.open("rb") as file:
        actual_hash = hashlib.file_digest(file, "sha256").hexdigest()

    if actual_hash != entry.sha256:
        raise ValueError(f"Checksum changed: {entry.source_file}")

print(f"Verified {len(manifest)} files: sizes and checksums match.") 

No extra CSV files found.


Verified 29 files: sizes and checksums match.


#### Read headers and count rows

One pass per file with the `csv` module:
- Record the column names and flag repeated, empty or padded names.
- Count data rows (header excluded), and flag blank records and records
  with the wrong number of fields.

In [4]:
def inspect_file(path):
    """Read the header and count the records of one CSV file."""
    with path.open(encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file, strict=True)
        columns = next(reader, None)

        if columns is None:
            raise ValueError(f"File is empty: {path.name}")

        rows = 0
        blank_rows = 0
        wrong_column_count = 0

        for record in reader:
            rows += 1
            if not record:
                blank_rows += 1
            if len(record) != len(columns):
                wrong_column_count += 1

    return {
        "column_count": len(columns),
        "columns": tuple(columns),
        "repeated_names": len(columns) != len(set(columns)),
        "empty_names": any(not name.strip() for name in columns),
        "padded_names": any(name != name.strip() for name in columns),
        "rows": rows,
        "blank_rows": blank_rows,
        "wrong_column_count": wrong_column_count,
    }

inventory_rows = []

for entry in manifest.itertuples(index=False):
    path = DATA_ROOT / entry.source_file

    # Stop and name the file if CSV parsing fails.
    try:
        details = inspect_file(path)
    except csv.Error as error:
        raise ValueError(
            f"CSV parsing failed: {entry.source_file}: {error}"
        ) from error

    inventory_rows.append({
        "source_folder": entry.source_folder,
        "source_file": entry.source_file,
        "file_name": path.name,
        **details,
    })

inventory = pd.DataFrame(inventory_rows)

# Later checks compare their row totals with these counts.
expected_rows = inventory.set_index("source_file")["rows"].to_dict()

display(inventory.drop(columns=["source_file", "columns"]))

,source_folder,file_name,column_count,repeated_names,empty_names,padded_names,rows,blank_rows,wrong_column_count
0,P1,dailyActivity_merged.csv,15,False,False,False,457,0,0
1,P1,heartrate_seconds_merged.csv,3,False,False,False,1154681,0,0
2,P1,hourlyCalories_merged.csv,3,False,False,False,24084,0,0
3,P1,hourlyIntensities_merged.csv,4,False,False,False,24084,0,0
4,P1,hourlySteps_merged.csv,3,False,False,False,24084,0,0
5,P1,minuteCaloriesNarrow_merged.csv,3,False,False,False,1445040,0,0
6,P1,minuteIntensitiesNarrow_merged.csv,3,False,False,False,1445040,0,0
7,P1,minuteMETsNarrow_merged.csv,3,False,False,False,1445040,0,0
8,P1,minuteSleep_merged.csv,4,False,False,False,198559,0,0
9,P1,minuteStepsNarrow_merged.csv,3,False,False,False,1445040,0,0


#### Compare headers across exports

Check whether matching filenames have the same column names and order.
This alone does not decide whether their data can be combined.

In [5]:
comparison_rows = []

for file_name, group in inventory.groupby("file_name"):
    columns_by_export = group.set_index("source_folder")["columns"].to_dict()

    if "P1" in columns_by_export and "P2" in columns_by_export:
        p1_columns = columns_by_export["P1"]
        p2_columns = columns_by_export["P2"]

        # Tuples compare names and order; sets compare names only.
        if p1_columns == p2_columns:
            result = "Same names and order"
        elif set(p1_columns) == set(p2_columns):
            result = "Same names, different order"
        else:
            result = "Different column names"
    else:
        result = "One export only"

    comparison_rows.append({
        "file_name": file_name,
        "exports": ", ".join(sorted(columns_by_export)),
        "header_check": result,
    })

header_comparison = pd.DataFrame(comparison_rows)
display(header_comparison)

,file_name,exports,header_check
0,dailyActivity_merged.csv,"P1, P2",Same names and order
1,dailyCalories_merged.csv,P2,One export only
2,dailyIntensities_merged.csv,P2,One export only
3,dailySteps_merged.csv,P2,One export only
4,heartrate_seconds_merged.csv,"P1, P2",Same names and order
5,hourlyCalories_merged.csv,"P1, P2",Same names and order
6,hourlyIntensities_merged.csv,"P1, P2",Same names and order
7,hourlySteps_merged.csv,"P1, P2",Same names and order
8,minuteCaloriesNarrow_merged.csv,"P1, P2",Same names and order
9,minuteCaloriesWide_merged.csv,P2,One export only


The 2.1 checks show:
- 29 files listed (P1: 11 files, 278.8 MB; P2: 18 files, 338.0 MB);
  sizes and checksums match the manifest, and there are no extra CSVs.
- No repeated, empty or padded column names; no blank records and no
  records with the wrong number of fields.
- 11 filenames appear in both exports, with identical column names and order.
- 7 appear only in P2. None appear only in P1.

### 2.2 Group files by their columns 

Files belong to the same header group when they have the same column names, regardless of order. 

In [6]:
# Sort names so column order does not affect grouping.
column_sets = inventory["columns"].map(lambda names: tuple(sorted(names)))
inventory["header_group"] = pd.factorize(column_sets, sort=False)[0] + 1

header_groups = inventory.groupby("header_group").agg(
    files=("source_file", "size"),
    column_count=("column_count", "first"),
    exports=("source_folder", lambda values: ", ".join(sorted(set(values)))),
    file_names=("file_name", lambda values: ", ".join(sorted(set(values)))),
)

display(header_groups)

,files,column_count,exports,file_names
header_group,,,,
1,2,15,"P1, P2",dailyActivity_merged.csv
2,2,3,"P1, P2",heartrate_seconds_merged.csv
3,2,3,"P1, P2",hourlyCalories_merged.csv
4,2,4,"P1, P2",hourlyIntensities_merged.csv
5,2,3,"P1, P2",hourlySteps_merged.csv
6,2,3,"P1, P2",minuteCaloriesNarrow_merged.csv
7,2,3,"P1, P2",minuteIntensitiesNarrow_merged.csv
8,2,3,"P1, P2",minuteMETsNarrow_merged.csv
9,2,4,"P1, P2",minuteSleep_merged.csv


### 2.3 Small previews

Read three records from the first file of each header group, as text.
Person IDs are replaced with preview-only codes (P01...), log IDs are
hidden, and empty values stay visible.

These records are not a representative sample.
Codes such as P01 are assigned separately in each preview.

In [7]:
def show_preview(file_info): 
    """Show three records with preview-only person codes and hidden log IDs."""
    shown = pd.read_csv(
        DATA_ROOT / file_info.source_file, nrows=3, **READ_POLICY
    )

    for column in shown.columns:
        values = shown[column]
        present = values.notna() & values.ne("")

        if column in PERSON_ID_COLUMNS:
            codes = {
                value: f"P{i + 1:02d}"
                for i, value in enumerate(pd.unique(values[present]))
            }
            shown[column] = values.replace(codes)
        elif column in LOG_ID_COLUMNS:
            shown[column] = values.mask(present, "[hidden]")

    print("Export:", file_info.source_folder)
    print("File  :", file_info.file_name)

    # Show fields vertically; wide files show only the first and last fields.
    if file_info.column_count > 20:
        with pd.option_context("display.max_rows", 12, "display.min_rows", 12):
            display(shown.T)
    else:
        display(shown.T)


for file_info in inventory.drop_duplicates("header_group").itertuples(index=False):
    show_preview(file_info)        

Export: P1
File  : dailyActivity_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityDate,3/25/2016,3/26/2016,3/27/2016
TotalSteps,11004,17609,12736
TotalDistance,7.1100001335144,11.5500001907349,8.52999973297119
TrackerDistance,7.1100001335144,11.5500001907349,8.52999973297119
LoggedActivitiesDistance,0,0,0
VeryActiveDistance,2.5699999332428,6.92000007629395,4.65999984741211
ModeratelyActiveDistance,0.46000000834465,0.730000019073486,0.159999996423721
LightActiveDistance,4.07000017166138,3.91000008583069,3.71000003814697
SedentaryActiveDistance,0,0,0


Export: P1
File  : heartrate_seconds_merged.csv


,0,1,2
Id,P01,P01,P01
Time,4/1/2016 7:54:00 AM,4/1/2016 7:54:05 AM,4/1/2016 7:54:10 AM
Value,93,91,96


Export: P1
File  : hourlyCalories_merged.csv


...

,0,1,2
Id,P01,P01,P01
ActivityHour,4/13/2016 12:00:00 AM,4/13/2016 1:00:00 AM,4/13/2016 2:00:00 AM
Intensity00,1,0,0
Intensity01,1,0,0
Intensity02,0,0,0
Intensity03,0,0,0
...,...,...,...
Intensity54,1,0,0
Intensity55,1,0,0
Intensity56,0,0,0


Export: P2
File  : minuteStepsWide_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityHour,4/13/2016 12:00:00 AM,4/13/2016 1:00:00 AM,4/13/2016 2:00:00 AM
Steps00,4,0,0
Steps01,16,0,0
Steps02,0,0,0
Steps03,0,0,0
...,...,...,...
Steps54,20,0,0
Steps55,1,0,0
Steps56,0,0,0


Export: P2
File  : sleepDay_merged.csv


,0,1,2
Id,P01,P01,P01
SleepDay,4/12/2016 12:00:00 AM,4/13/2016 12:00:00 AM,4/15/2016 12:00:00 AM
TotalSleepRecords,1,2,1
TotalMinutesAsleep,327,384,412
TotalTimeInBed,346,407,442


The previews show:
- **Daily activity (group 1):** all three records share the same ID code.
  Dates use month/day/year. Long decimals and zero values remain unchanged.
  Units and the meaning of Calories still need checking.
- **Heart rate (group 2):** all three records share the same ID code.
  Time includes seconds and AM/PM. These three timestamps are 5 seconds
  apart; that doesn't establish the spacing throughout the file.
  The meaning and unit of Value still need checking.
- **Hourly files (groups 3-5):** ActivityHour includes a date and a 12-hour
  time. The steps and intensity previews contain zeros; we keep them as
  recorded. The calories preview contains 48; its meaning still needs checking.
- **Minute narrow files (groups 6-8, 10):** decimal calorie values and METs
  values of 10. Meanings, units, and scaling still need checking.
- **Sleep and weight (groups 9, 11):** sleep timestamps include seconds
  ending in :30. Sleep value is 1 in these rows; its meaning is still unknown.
  Fat has an empty field; we'll include "" in its missing-token list.
  IsManualReport shows True and False, currently stored as text.
  The ID masking keeps empty values visible.
- **Daily files only in P2 (groups 12-14, 18):** ActivityDay is date only.
  SleepDay has a date plus time, shown at midnight here; that doesn't
  establish when sleep started. TotalSleepRecords includes 2; we still need
  its definition.
- **Wide files (groups 15-17):** 62 columns: ActivityHour and numbered fields
  from 00 to 59. These suggest minute slots; verify before reshaping or
  comparing with narrow files.

P01 is assigned separately in each preview. It doesn't prove that
files show the same person.

### 2.4 Check and parse dates

Map each date column to a format seen in the previews.
Then check that format against the full files, keeping the original text.

In [8]:
# Formats seen in the previews; the full-file check follows.
date_only = "%m/%d/%Y"
date_and_time = "%m/%d/%Y %I:%M:%S %p"

DATE_FORMATS = {
    "ActivityDate": date_only,
    "ActivityDay": date_only,
    "ActivityHour": date_and_time,
    "ActivityMinute": date_and_time,
    "Time": date_and_time,
    "date": date_and_time,
    "Date": date_and_time,
    "SleepDay": date_and_time,
}

date_rows = []

for file_info in inventory.itertuples(index=False):
    matched_columns = [c for c in file_info.columns if c in DATE_FORMATS]

    if not matched_columns:
        raise ValueError(f"No date column mapped: {file_info.file_name}")

    for column in matched_columns:
        date_rows.append({
            "source_folder": file_info.source_folder,
            "source_file": file_info.source_file,
            "file_name": file_info.file_name,
            "date_column": column,
            "format": DATE_FORMATS[column],
        })

date_plan = pd.DataFrame(date_rows)
display(date_plan.drop(columns=["source_file"]))

,source_folder,file_name,date_column,format
0,P1,dailyActivity_merged.csv,ActivityDate,%m/%d/%Y
1,P1,heartrate_seconds_merged.csv,Time,%m/%d/%Y %I:%M:%S %p
2,P1,hourlyCalories_merged.csv,ActivityHour,%m/%d/%Y %I:%M:%S %p
3,P1,hourlyIntensities_merged.csv,ActivityHour,%m/%d/%Y %I:%M:%S %p
4,P1,hourlySteps_merged.csv,ActivityHour,%m/%d/%Y %I:%M:%S %p
5,P1,minuteCaloriesNarrow_merged.csv,ActivityMinute,%m/%d/%Y %I:%M:%S %p
6,P1,minuteIntensitiesNarrow_merged.csv,ActivityMinute,%m/%d/%Y %I:%M:%S %p
7,P1,minuteMETsNarrow_merged.csv,ActivityMinute,%m/%d/%Y %I:%M:%S %p
8,P1,minuteSleep_merged.csv,date,%m/%d/%Y %I:%M:%S %p
9,P1,minuteStepsNarrow_merged.csv,ActivityMinute,%m/%d/%Y %I:%M:%S %p


In [9]:
def check_file_dates(rule):
    """Parse one date column in chunks; count empty and failed dates."""
    counts = {
        "rows": 0,
        "empty_dates": 0,
        "eligible_dates": 0,
        "parsed_dates": 0,
        "failed_dates": 0,
    }
    minimums = []
    maximums = []
    examples = []

    with read_chunks(rule, [rule.date_column]) as chunks:
        for chunk in chunks:
            raw = chunk[rule.date_column]
            non_empty = raw.notna() & raw.ne("")

            # Parse into a separate Series; the original text is unchanged.
            parsed = pd.to_datetime(
                raw, format=rule.format, exact=True, errors="coerce"
            )
            failed = non_empty & parsed.isna()
            valid_dates = parsed[non_empty & parsed.notna()]

            counts["rows"] += len(chunk)
            counts["empty_dates"] += int((~non_empty).sum())
            counts["eligible_dates"] += int(non_empty.sum())
            counts["parsed_dates"] += len(valid_dates)
            counts["failed_dates"] += int(failed.sum())

            if not valid_dates.empty:
                minimums.append(valid_dates.min())
                maximums.append(valid_dates.max())

            # Keep at most five different failing values per file.
            for value in raw[failed].drop_duplicates():
                if len(examples) == 5:
                    break
                if value not in examples:
                    examples.append(value)

    return {
        "source_folder": rule.source_folder,
        "source_file": rule.source_file,
        "file_name": rule.file_name,
        "date_column": rule.date_column,
        **counts,
        "failure_percent": percent(counts["failed_dates"], counts["eligible_dates"]),
        "min_date": min(minimums, default=pd.NaT),
        "max_date": max(maximums, default=pd.NaT),
        "failed_examples": examples,
    }


date_results = []

for rule in date_plan.itertuples(index=False):
    result = check_file_dates(rule)

    # This pass must read the rows counted in 2.1.
    if result["rows"] != expected_rows[rule.source_file]:
        raise ValueError(f"Row count differs: {rule.file_name}")

    # Print examples only when parsing fails.
    if result["failed_examples"]:
        print(rule.source_folder, rule.file_name)
        for value in result["failed_examples"]:
            print(repr(value))

    date_results.append(result)

date_report = pd.DataFrame(date_results)
display(date_report.drop(columns=["source_file", "failed_examples"]))

,source_folder,file_name,date_column,rows,empty_dates,eligible_dates,parsed_dates,failed_dates,failure_percent,min_date,max_date
0,P1,dailyActivity_merged.csv,ActivityDate,457,0,457,457,0,0.0,2016-03-12 00:00:00,2016-04-12 00:00:00
1,P1,heartrate_seconds_merged.csv,Time,1154681,0,1154681,1154681,0,0.0,2016-03-29 00:00:05,2016-04-12 11:03:15
2,P1,hourlyCalories_merged.csv,ActivityHour,24084,0,24084,24084,0,0.0,2016-03-12 00:00:00,2016-04-12 10:00:00
3,P1,hourlyIntensities_merged.csv,ActivityHour,24084,0,24084,24084,0,0.0,2016-03-12 00:00:00,2016-04-12 10:00:00
4,P1,hourlySteps_merged.csv,ActivityHour,24084,0,24084,24084,0,0.0,2016-03-12 00:00:00,2016-04-12 10:00:00
5,P1,minuteCaloriesNarrow_merged.csv,ActivityMinute,1445040,0,1445040,1445040,0,0.0,2016-03-12 00:00:00,2016-04-12 10:59:00
6,P1,minuteIntensitiesNarrow_merged.csv,ActivityMinute,1445040,0,1445040,1445040,0,0.0,2016-03-12 00:00:00,2016-04-12 10:59:00
7,P1,minuteMETsNarrow_merged.csv,ActivityMinute,1445040,0,1445040,1445040,0,0.0,2016-03-12 00:00:00,2016-04-12 10:59:00
8,P1,minuteSleep_merged.csv,date,198559,0,198559,198559,0,0.0,2016-03-11 21:19:30,2016-04-12 08:35:00
9,P1,minuteStepsNarrow_merged.csv,ActivityMinute,1445040,0,1445040,1445040,0,0.0,2016-03-12 00:00:00,2016-04-12 10:59:00


Every date column was parsed in full: no empty dates and no parsing
failures. For example, all 457 P1 daily-activity dates parsed
(March 12-April 12, 2016).

### 2.5 Observed date ranges

The ranges come from full-file checks, not samples.

Important findings:
- Daily activity includes April 12 in both exports.
  This does not yet prove duplicate participant records.
- P1 minute sleep starts on March 11; P2 minute sleep starts on April 11,
  before their folder start labels.
- P2 wide files span April 13-May 13.
  P2 narrow minute files span April 12-May 12.
  The reason for this difference is not yet known.
- Coverage differs by file. For example, P1 heart-rate data starts
  on March 29, while daily activity starts on March 12.

Keep the observed timestamps unchanged.
Do not trim or shift dates to match folder labels.

Successful parsing does not prove that timestamps are correctly aligned.
A minimum and maximum do not prove continuous coverage.
The source timezone remains unknown.

**2.6: File-set differences are covered by the header comparison: 7 files
exist only in P2 (structural absence, not missing values).**

## Step 3: Field dictionary

Steps 3.1 and 3.2 share one full read of every file (chunks, values as text):
- For every column: empty and non-empty counts, and the first non-empty value.
- For every column except IDs and dates: a count of every value, kept in
  `value_counts` so later checks can reuse it without another read.
- Totals are checked against the 2.1 row counts.

Here, empty means `""`. Other possible missing tokens are checked later.

In [10]:
# Reuse the date columns from 2.4.
date_fields = set(zip(date_plan["source_file"], date_plan["date_column"]))

column_profiles = []  # one entry per file, export and column
value_counts = {}     # (export, file_name, column) -> Counter of every value

for file_info in inventory.itertuples(index=False):
    profiles = {
        column: {"empty": 0, "non_empty": 0, "example": None}
        for column in file_info.columns
    }
    counters = {
        column: Counter()
        for column in file_info.columns
        if column not in ID_COLUMNS
        and (file_info.source_file, column) not in date_fields
    }

    with read_chunks(file_info) as chunks:
        for chunk in chunks:
            for column, profile in profiles.items():
                values = chunk[column]

                if values.isna().any():
                    raise ValueError(
                        f"Unexpected null: {file_info.file_name}, {column}"
                    )

                present = values[values.ne("")]
                profile["empty"] += len(values) - len(present)
                profile["non_empty"] += len(present)

                if profile["example"] is None and not present.empty:
                    profile["example"] = present.iloc[0]

                if column in counters:
                    counters[column].update(values.value_counts().to_dict())

    for column, profile in profiles.items():
        # Check that every record was counted.
        if profile["empty"] + profile["non_empty"] != expected_rows[file_info.source_file]:
            raise ValueError(f"Count differs: {file_info.file_name}, {column}")

        column_profiles.append({
            "source_folder": file_info.source_folder,
            "source_file": file_info.source_file,
            "file_name": file_info.file_name,
            "column": column,
            **profile,
        })

        if column in counters:
            key = (file_info.source_folder, file_info.file_name, column)
            value_counts[key] = counters[column]

print("Columns profiled:", len(column_profiles))
print("Columns with value counts:", len(value_counts))

Columns profiled: 311
Columns with value counts: 249


### 3.1 Observed dictionary

For every column in every file and export: value format, empty/non-empty
counts and one original example. IDs stay text (original IDs may be shown,
rule 0.6); date columns reuse the 2.4 checks. Each column gets a short
meaning guess, marked **unverified**.

The type labels describe the text format, not a confirmed physical meaning.

In [11]:
MEANING_GUESSES = {
    "Id": "Participant identifier",
    "ActivityDate": "Date of the daily activity record",
    "ActivityDay": "Date of the daily record",
    "ActivityHour": "Hour linked to the record",
    "ActivityMinute": "Minute linked to the record",
    "Time": "Time linked to the heart-rate reading",
    "date": "Time linked to the sleep record",
    "Date": "Time linked to the weight record",
    "SleepDay": "Date assigned to the sleep summary",
    "TotalSteps": "Daily step total",
    "StepTotal": "Step total for the record's period",
    "Steps": "Steps for the record's period",
    "TotalDistance": "Total distance for the record's period",
    "TrackerDistance": "Distance recorded by the tracker",
    "LoggedActivitiesDistance": "Distance linked to logged activities",
    "VeryActiveDistance": "Distance labelled very active",
    "ModeratelyActiveDistance": "Distance labelled moderately active",
    "LightActiveDistance": "Distance labelled lightly active",
    "SedentaryActiveDistance": "Distance linked to the sedentary category",
    "VeryActiveMinutes": "Minutes labelled very active",
    "FairlyActiveMinutes": "Minutes labelled fairly active",
    "LightlyActiveMinutes": "Minutes labelled lightly active",
    "SedentaryMinutes": "Minutes labelled sedentary",
    "TotalIntensity": "Total activity intensity",
    "AverageIntensity": "Average activity intensity",
    "Intensity": "Activity intensity value or code",
    "METs": "MET-related activity measure; scaling unknown",
    "Value": "Heart-rate reading; unit unconfirmed",
    "value": "Sleep-related value or code; meaning unknown",
    "logId": "Sleep log identifier",
    "LogId": "Weight log identifier",
    "WeightKg": "Weight labelled in kilograms",
    "WeightPounds": "Weight labelled in pounds",
    "Fat": "Body-fat measure; unit unknown",
    "BMI": "Body mass index value",
    "IsManualReport": "Whether the record was entered manually",
    "TotalSleepRecords": "Number of sleep records in the daily summary",
    "TotalMinutesAsleep": "Daily summary of minutes asleep",
    "TotalTimeInBed": "Daily summary of time in bed; unit unconfirmed",
}

CALORIE_PERIODS = {
    "dailyActivity_merged.csv": "day",
    "dailyCalories_merged.csv": "day",
    "hourlyCalories_merged.csv": "hour",
    "minuteCaloriesNarrow_merged.csv": "minute",
}


def guess_meaning(column, file_name):
    """Short, unverified meaning guess from the column and file name."""
    # Make the calorie period clear from the file.
    if column == "Calories" and file_name in CALORIE_PERIODS:
        period = CALORIE_PERIODS[file_name]
        return (
            f"Calorie value for the labelled {period}; "
            "exact definition and unit unverified"
        )

    # Apply one shared explanation to numbered wide fields.
    for prefix in ("Calories", "Intensity", "Steps"):
        suffix = column.removeprefix(prefix)
        if column.startswith(prefix) and len(suffix) == 2 and suffix.isdigit():
            return (
                f"{prefix} value for slot {suffix} within the "
                "recorded hour; slot duration unknown"
            )

    return MEANING_GUESSES.get(column, "Meaning unknown")

In [12]:
PATTERNS = {
    "integer": r"[+-]?[0-9]+",
    "decimal": r"[+-]?[0-9]+\.[0-9]+",
    "true_false": r"True|False",
}


def count_formats(counter):
    """Count records matching each text pattern, using the value counts."""
    found = dict.fromkeys(PATTERNS, 0)
    for value, count in counter.items():
        for name, pattern in PATTERNS.items():
            if re.fullmatch(pattern, value):
                found[name] += count
    return found


dictionary_rows = []

for profile in column_profiles:
    column = profile["column"]
    key = (profile["source_folder"], profile["file_name"], column)
    is_id = column in ID_COLUMNS
    is_date = (profile["source_file"], column) in date_fields

    # IDs and dates are not format-checked here (no value counts).
    formats = (
        count_formats(value_counts[key])
        if key in value_counts
        else dict.fromkeys(PATTERNS, 0)
    )
    non_empty = profile["non_empty"]

    if non_empty == 0:
        inferred = "all empty"
    elif is_id:
        inferred = "identifier (text)"
    elif is_date:
        inferred = "date (checked in 2.4)"
    elif formats["integer"] == non_empty:
        inferred = "integer"
    elif formats["integer"] + formats["decimal"] == non_empty:
        inferred = "decimal"
    elif formats["true_false"] == non_empty:
        inferred = "true/false"
    else:
        inferred = "mixed / text"

    dictionary_rows.append({
        "source_folder": profile["source_folder"],
        "source_file": profile["source_file"],
        "file_name": profile["file_name"],
        "column": column,
        "inferred_type": inferred,
        "empty": profile["empty"],
        "non_empty": non_empty,
        **formats,
        "example": profile["example"],
        "other_text": (
            pd.NA if is_id or is_date else non_empty - sum(formats.values())
        ),
        "meaning_guess": guess_meaning(column, profile["file_name"]),
        "meaning_status": "unverified",
    })

observed_dictionary = pd.DataFrame(dictionary_rows)

print("Dictionary entries:", len(observed_dictionary))
print("Distinct column names:", observed_dictionary["column"].nunique())

display(observed_dictionary[[
    "source_folder",
    "file_name",
    "column",
    "inferred_type",
    "non_empty",
    "empty",
    "example",
    "meaning_guess",
    "meaning_status",
]])

Dictionary entries:

 311
Distinct column names: 220


,source_folder,file_name,column,inferred_type,non_empty,empty,example,meaning_guess,meaning_status
0,P1,dailyActivity_merged.csv,Id,identifier (text),457,0,1503960366,Participant identifier,unverified
1,P1,dailyActivity_merged.csv,ActivityDate,date (checked in 2.4),457,0,3/25/2016,Date of the daily activity record,unverified
2,P1,dailyActivity_merged.csv,TotalSteps,integer,457,0,11004,Daily step total,unverified
3,P1,dailyActivity_merged.csv,TotalDistance,decimal,457,0,7.1100001335144,Total distance for the record's period,unverified
4,P1,dailyActivity_merged.csv,TrackerDistance,decimal,457,0,7.1100001335144,Distance recorded by the tracker,unverified
...,...,...,...,...,...,...,...,...,...
306,P2,weightLogInfo_merged.csv,WeightPounds,decimal,67,0,115.963146545323,Weight labelled in pounds,unverified
307,P2,weightLogInfo_merged.csv,Fat,integer,2,65,22,Body-fat measure; unit unknown,unverified
308,P2,weightLogInfo_merged.csv,BMI,decimal,67,0,22.6499996185303,Body mass index value,unverified
309,P2,weightLogInfo_merged.csv,IsManualReport,true/false,67,0,True,Whether the record was entered manually,unverified


In [13]:
# Compare the same filename and column across exports.
type_comparison = observed_dictionary.pivot(
    index=["file_name", "column"],
    columns="source_folder",
    values="inferred_type",
)

shared_fields = type_comparison.dropna(subset=["P1", "P2"])
different_types = shared_fields.loc[shared_fields["P1"].ne(shared_fields["P2"])]

# Review mixed formats, all-empty columns and partly empty columns.
needs_review = observed_dictionary.loc[
    observed_dictionary["inferred_type"].isin(["mixed / text", "all empty"])
    | observed_dictionary["empty"].gt(0),
    [
        "source_folder",
        "file_name",
        "column",
        "inferred_type",
        "non_empty",
        "empty",
        "other_text",
    ],
]

print("Shared fields with different types:")
display(different_types)

print("Columns needing review:")
display(needs_review)

Shared fields with different types:


,source_folder,P1,P2
file_name,column,,


Columns needing review:


,source_folder,file_name,column,inferred_type,non_empty,empty,other_text
48,P1,weightLogInfo_merged.csv,Fat,integer,2,31,0
307,P2,weightLogInfo_merged.csv,Fat,integer,2,65,0


**Saved results:** 311 file/export/column entries and 220 distinct column
names. Shared fields have matching types across P1 and P2. The empty-value
review shows only `Fat`: P1 has 31 empty values out of 33; P2 has 65 out
of 67. There are no mixed/text or all-empty columns.

See [data notes, section 7](../docs/data_notes.md#7-files-columns-and-guessed-meanings)
for **Column | Guessed meaning | Record 1 | Record 2** tables, with each
export separate. The two examples are not representative samples. Only
wide slots `00`-`02` are displayed there; full-file checks still include
every column.

Possible repeated measures are listed for Step 6.3; they are not confirmed
duplicates. The [PDF reference](../docs/Fitbit_Field_Reference.pdf) is a
sharing copy; the data notes are the main record for later wording changes.

Reviewed on 2026-10-19. Meaning guesses remain unverified.

### 3.2 Distinct values

Uses `value_counts` from the Step 3 read (IDs and date columns skipped).
- Count distinct non-empty values; count empty strings separately.
- Flag columns with one non-empty value, and columns where one value
  covers at least 95% of non-empty records. The 95% threshold only helps
  review; it does not justify removing a column.
- Save every value and its count for columns with 20 or fewer distinct
  non-empty values to `EDA/distinct_value_counts.csv`.
- Values stay as original text: `"0"` and `"0.0"` are different values.

In [14]:
summary_rows = []
detail_rows = []

for (export, file_name, column), counts in value_counts.items():
    # Sort by count, then by text, so ties have a fixed order.
    ranked = sorted(
        ((value, count) for value, count in counts.items() if value != ""),
        key=lambda item: (-item[1], item[0]),
    )
    non_empty = sum(count for _, count in ranked)
    distinct = len(ranked)
    top_value, top_count = ranked[0] if ranked else (pd.NA, 0)
    share = top_count / non_empty if non_empty else None

    summary_rows.append({
        "source_folder": export,
        "file_name": file_name,
        "column": column,
        "distinct_non_empty": distinct,
        "empty": counts.get("", 0),
        "non_empty": non_empty,
        "top_value": top_value,
        "top_count": top_count,
        "top_percent": percent(top_count, non_empty),
        "one_non_empty_value": distinct == 1,
        "mostly_one_value": share >= 0.95 if share is not None else pd.NA,
    })

    # Keep every value, including empty strings, for short value lists.
    if distinct <= 20:
        for value, count in sorted(counts.items(), key=lambda item: (-item[1], item[0])):
            detail_rows.append({
                "source_folder": export,
                "file_name": file_name,
                "column": column,
                "value": value,
                "is_empty": value == "",
                "count": count,
            })

value_summary = pd.DataFrame(summary_rows)
small_value_counts = pd.DataFrame(detail_rows)
small_value_counts.to_csv(COUNTS_PATH, index=False)

display(value_summary)
print("Detailed counts saved:", COUNTS_PATH)

,source_folder,file_name,column,distinct_non_empty,empty,non_empty,top_value,top_count,top_percent,one_non_empty_value,mostly_one_value
0,P1,dailyActivity_merged.csv,TotalSteps,389,0,457,0,61,13.35,False,False
1,P1,dailyActivity_merged.csv,TotalDistance,334,0,457,0,63,13.79,False,False
2,P1,dailyActivity_merged.csv,TrackerDistance,336,0,457,0,66,14.44,False,False
3,P1,dailyActivity_merged.csv,LoggedActivitiesDistance,20,0,457,0,433,94.75,False,False
4,P1,dailyActivity_merged.csv,VeryActiveDistance,170,0,457,0,245,53.61,False,False
...,...,...,...,...,...,...,...,...,...,...,...
244,P2,weightLogInfo_merged.csv,WeightKg,34,0,67,61.2000007629395,5,7.46,False,False
245,P2,weightLogInfo_merged.csv,WeightPounds,34,0,67,134.922906139214,5,7.46,False,False
246,P2,weightLogInfo_merged.csv,Fat,2,65,2,22,1,50.00,False,False
247,P2,weightLogInfo_merged.csv,BMI,36,0,67,23.8899993896484,5,7.46,False,False


Detailed counts saved: D:\Projects\AI and ML\FitBit Fitness Tracker Data\EDA\distinct_value_counts.csv


In [15]:
# Columns to review: few distinct values, one value, or one value >= 95%.
few_values = value_summary["distinct_non_empty"].le(20)
flagged = value_summary["one_non_empty_value"] | value_summary["mostly_one_value"].eq(True)

print("Columns with value counts:", len(value_summary))
print("With 20 or fewer distinct non-empty values:", int(few_values.sum()))
print("With one non-empty value:", int(value_summary["one_non_empty_value"].sum()))
print("With one value covering >= 95%:", int(value_summary["mostly_one_value"].eq(True).sum()))

with pd.option_context("display.max_rows", 200):
    display(value_summary.loc[few_values | flagged])

Columns with value counts: 249
With 20 or fewer distinct non-empty values: 74
With one non-empty value: 0
With one value covering >= 95%: 1


,source_folder,file_name,column,distinct_non_empty,empty,non_empty,top_value,top_count,top_percent,one_non_empty_value,mostly_one_value
3,P1,dailyActivity_merged.csv,LoggedActivitiesDistance,20,0,457,0,433,94.75,False,False
7,P1,dailyActivity_merged.csv,SedentaryActiveDistance,8,0,457,0,419,91.68,False,False
19,P1,minuteIntensitiesNarrow_merged.csv,Intensity,4,0,1445040,0,1234832,85.45,False,False
21,P1,minuteSleep_merged.csv,value,3,0,198559,1,183526,92.43,False,False
25,P1,weightLogInfo_merged.csv,Fat,2,31,2,10,1,50.00,False,False
27,P1,weightLogInfo_merged.csv,IsManualReport,2,0,33,True,23,69.70,False,False
31,P2,dailyActivity_merged.csv,LoggedActivitiesDistance,19,0,940,0,908,96.60,False,True
35,P2,dailyActivity_merged.csv,SedentaryActiveDistance,9,0,940,0,858,91.28,False,False
46,P2,dailyIntensities_merged.csv,SedentaryActiveDistance,9,0,940,0,858,91.28,False,False
117,P2,minuteIntensitiesNarrow_merged.csv,Intensity,4,0,1325580,0,1112102,83.90,False,False


**3.2 observations** (from the outputs above; meanings stay unverified):
- 249 columns were counted (IDs and dates skipped); 74 have 20 or fewer
  distinct non-empty values.
- No column has only one non-empty value.
- One column reaches the 95% flag: P2 `dailyActivity` `LoggedActivitiesDistance`,
  where `0` is 908 of 940 values (96.60%). In P1 the same column is just
  below the flag (433 of 457, 94.75%).
- **Possible codes** (small sets of whole numbers): `Intensity` in both narrow
  files and in every wide slot (`0`-`3`), and sleep `value` in both exports
  (`1`-`3`). `TotalSleepRecords` takes `1`-`3` (a count or a code). What each
  number means is unknown; averages of codes are not a physical scale.
- `IsManualReport` holds only `True` and `False`.
- `Fat` has two non-empty values per export (P1: `10`, `22`; P2: `22`, `25`);
  all other records are empty.
- `LoggedActivitiesDistance` and `SedentaryActiveDistance` have few values,
  mostly `0`. They look like measurements, not codes; ranges come in 3.3.
- No values or columns were changed or removed.